In [1]:
import os
import deepchem as dc
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score  # Для F1-macro
from data.preprocessing.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from rdkit import Chem
from deepchem.data import InMemoryLoader

from sklearn.utils.class_weight import compute_class_weight

from deepchem.models.optimizers import ExponentialDecay
from deepchem.hyper import HyperparamOpt

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


c:\Users\edtop\ITMO\thotex_hack\thortex-hackaton\.venv\Lib\site-packages\deepchem\models\torch_models\layers.py:1207: SyntaxWarning: invalid escape sequence '\m'
  Input node features of shape :math:`(|\mathcal{V}|, F_n)`
c:\Users\edtop\ITMO\thotex_hack\thortex-hackaton\.venv\Lib\site-packages\deepchem\models\torch_models\attention.py:64: SyntaxWarning: invalid escape sequence '\i'
  Given $X\in \mathbb{R}^{n \times in_feature}$, the attention is calculated by: $a=softmax(W_2tanh(W_1X))$, where
c:\Users\edtop\ITMO\thotex_hack\thortex-hackaton\.venv\Lib\site-packages\deepchem\models\torch_models\attention.py:96: SyntaxWarning: invalid escape sequence '\m'
  input feature of shape $\mathbb{R}^{n \times in_feature}$.


ImportError: cannot import name 'Mapping' from 'collections' (C:\Users\edtop\AppData\Roaming\uv\python\cpython-3.13.3-windows-x86_64-none\Lib\collections\__init__.py)

In [ ]:
pipeline1 = Pipeline(feature_type="rdkit+dft", 
                    use_pca_rdkit=True, 
                    use_pca_dft=False, 
                    remove_zero_var=True)

train_df, _ = pipeline1.get_processed_data()

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
Extracting raw features:   0%|          | 9/2064 [00:00<00:24, 83.88it/s][18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use MorganGenerator
[18:14:44] DEPRECATION WARNING: please use

----

In [ ]:
tasks = ['taste_cluster']
metric = dc.metrics.Metric(f1_macro, mode='classification')
# Определение дополнительных колонок (DFT + RDKit, все 67 колонок после SMILES)
extra_columns = [col for col in train_df.columns if col not in ['Pubchem_ID', 'taste_cluster', 'SMILES']]

# Featurizer для SMILES: CircularFingerprint (ECFP, размер 1024)
featurizer = dc.feat.CircularFingerprint(size=2048)

train_df.to_csv('temp_data.csv', index=False)

# Загрузка датасета с SMILES (автоматически пропустит невалидные)
loader = dc.data.CSVLoader(tasks=tasks, feature_field='SMILES', featurizer=featurizer)
dataset = loader.create_dataset('temp_data.csv')

# Извлечение extra_feats (DFT + RDKit) только для валидных строк (где featurization удался)
valid_mask = [i for i in range(len(dataset.X)) if dataset.X[i].size > 0]  # Только непустые X
extra_feats = train_df.iloc[valid_mask][extra_columns].values.astype(float)  # numpy array, обработаем NaN ниже

[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerator
[18:16:19] DEPRECATION WARNING: please use MorganGenerat

In [63]:
def f1_macro(y_true, y_pred):
    y_true_labels = np.argmax(y_true, axis=-1).ravel()
    y_pred_labels = np.argmax(y_pred, axis=-1).ravel()
    return f1_score(y_true_labels, y_pred_labels, average='macro')

metric = dc.metrics.Metric(
    f1_macro,
    mode='classification',
    classification_handling_mode='direct'
)

In [68]:
# Обработка NaN в extra_feats (заменим на 0 или среднее; здесь на 0 для простоты)
extra_feats = np.nan_to_num(extra_feats, nan=0.0)

# Нормализация extra_feats (важно для ML)
#scaler = StandardScaler()
#extra_feats = scaler.fit_transform(extra_feats)

# Конкатенация X (ECFP) с extra_feats
X_ecfp = np.array([x for x in dataset.X if x.size > 0])  # Только валидные
X_combined = np.hstack((X_ecfp, extra_feats))

y_raw = dataset.y[valid_mask].flatten()  # (N,)
print('Raw unique:', np.unique(y_raw))   # Должно быть [0. 1. 2. 3. 4. 5. 6. 7.]

# Округляем на случай погрешностей float и приводим к int
y_raw = np.rint(y_raw).astype(int)

# Ремэппим в consecutive 0..k-1 (на случай gaps в будущем)
le = LabelEncoder()
y = le.fit_transform(y_raw).reshape(-1, 1)  # (N, 1) — обязательно!
n_classes = len(le.classes_)
print('Encoded unique (int):', np.unique(y))
print('n_classes =', n_classes)  # Должно быть 8

class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y.flatten())
class_weight_dict = dict(zip(np.unique(y), class_weights))

# В fit: model.fit(train_dataset, nb_epoch=200, sample_weight=train_dataset.w)  # Но нужно создать w
w = np.array([class_weight_dict[label] for label in y.flatten()])
combined_dataset = dc.data.NumpyDataset(X=X_combined, y=y.reshape(-1, 1), w=w.reshape(-1, 1), ids=dataset.ids[valid_mask])

# Датасет
#combined_dataset = dc.data.NumpyDataset(X=X_combined, y=y, ids=dataset.ids[valid_mask])

# Разделение на train/valid/test (80/10/10)
splitter = dc.splits.RandomSplitter()
train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(combined_dataset)
n_features = X_combined.shape[1]

Raw unique: [0. 1. 2. 3. 4. 5. 6. 7.]
Encoded unique (int): [0 1 2 3 4 5 6 7]
n_classes = 8


In [78]:
# Убедитесь, что n_features правильно вычислено
print(f"n_features: {n_features}")
print(f"X shape: {train_dataset.X.shape}")

n_features: 2115
X shape: (1651, 2115)


In [ ]:
from deepchem.hyper import GridHyperparamOpt  # или RandomSearchHyperparamOpt
n_classes = 8
# Создаем функцию-билдер модели
def model_builder(**model_params):
    # Базовые параметры + параметры для тюнинга
    all_params = {
        'n_tasks': 1,
        'n_features': n_features,
        'mode': 'classification',
        'n_classes': 8,
        **model_params  # Добавляем параметры для тюнинга
    }
    return dc.models.MultitaskClassifier(**all_params)

# Параметры для тюнинга
params_dict = {
    'layer_sizes': [[256, 128], [512, 256, 128], [1024, 512]],
    'dropout': [0.1, 0.2, 0.3],
    'learning_rate': [0.001, 0.0005]
}

# Создаем оптимизатор с функцией-билдером
optimizer = GridHyperparamOpt(model_builder)  # или RandomSearchHyperparamOpt

# Запускаем поиск
best_model, best_params, all_results = optimizer.hyperparam_search(
    params_dict, 
    train_dataset, 
    valid_dataset, 
    metric
)

ValueError: y has more than n_class unique elements.

In [ ]:
# Модель: MultitaskClassifier для классификации  # Размер после конкатенации
model = dc.models.MultitaskClassifier(n_tasks=1, n_features=n_features, layer_sizes=[512, 256],
                                      mode='classification', n_classes=n_classes, dropouts=0.2)

# Обучение (epochs=100, настройте)
model.fit(train_dataset, nb_epoch=200)

0.21113927841186522

In [ ]:
# !!! КЛЮЧЕВОЕ: передаём n_classes !!!
valid_score = model.evaluate(valid_dataset, [metric], n_classes=n_classes)
print('Valid F1-macro:', valid_score['f1_macro'])

Valid F1-macro: 0.39461085202902535


In [ ]:
# Предсказание для новой молекулы (пример)
new_smiles = 'CCO'
new_ecfp = featurizer.featurize([new_smiles])[0]
# Для extra_feats новой молекулы: нужно посчитать DFT/RDKit отдельно; здесь используем zeros как placeholder
new_extra = np.zeros((1, len(extra_columns)))
#new_extra = scaler.transform(new_extra)  # Нормализуем
new_X = np.hstack((new_ecfp.reshape(1, -1), new_extra))
new_dataset = dc.data.NumpyDataset(new_X)
prediction = model.predict(new_dataset)
print('Prediction:', prediction)